#### Task 1: Simple vector embedding generation

**Objective:**
Generate vector embeddings from text data.

**Task Description:**

- load embedding model into ollama:
    1. Attach ollama container shell
    2. Run command in container `ollama pull pull bge-m3` or send an api call `requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False}` via the requests library to download embedding model.
- embed simple text queries

How to select the right embedding model: [MTEB - Massive Text Embedding Benchmark](https://huggingface.co/blog/mteb)

**Useful links:**

- [Ollama REST API](https://www.postman.com/postman-student-programs/ollama-api/documentation/suc47x8/ollama-rest-api)
- [Langchain Ollama Embeddings](https://python.langchain.com/docs/integrations/text_embedding/ollama/)
- [Langchain Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/)


In [43]:
import requests

In [44]:
# Download embedding model into ollama container
requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False})

<Response [200]>

In [45]:
# List all available models in the ollama container
# Note: /api/ps shows only running models, /api/tags shows all installed models
response = requests.get("http://localhost:11434/api/tags")
response.json()

{'models': [{'name': 'bge-m3:latest',
   'model': 'bge-m3:latest',
   'modified_at': '2026-04-22T17:10:16.176581469Z',
   'size': 1157672605,
   'digest': '7907646426070047a77226ac3e684fbbe8410524f7b4a74d02837e43f2146bab',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'bert',
    'families': ['bert'],
    'parameter_size': '566.70M',
    'quantization_level': 'F16'}},
  {'name': 'qwen3.5:2b',
   'model': 'qwen3.5:2b',
   'modified_at': '2026-04-09T08:53:56.81333625Z',
   'size': 2741192820,
   'digest': '324d162be6ca5629ae4517c8710434d0bd2d665bc94dbad46e9af8fbf8a2f0df',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'qwen35',
    'families': ['qwen35'],
    'parameter_size': '2.3B',
    'quantization_level': 'Q8_0'}}]}

In [46]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

In [47]:
text = "This is a test document."

# ADD HERE YOUR CODE
# Perform vector search
query_vector = embedding_model.embed_query(text)

print(f"Embedding vector length: {len(query_vector)}")
print(query_vector[:10])

Embedding vector length: 1024
[-0.050007477, 0.021129724, -0.057080247, -0.005526018, -0.0062946486, -0.028408367, 0.0451794, 0.042706486, -0.00021894443, 0.013056353]


#### Task 2: Generate embedding vectors with custom dataset

**Objective:**
Load custom dataset, preprocess it and generate vector embeddings.

**Task Description:**

- load pdf document "AI_Book.pdf" via langchain document loader: ``PyPDFLoader``
- use RecursiveCharacterTextSplitter to split documents into chunks
- generate embeddings for single documents

**RecursiveCharacterTextSplitter:**

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is `["\n\n", "\n", " ", ""]`. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

**Useful links:**

- [Langchain PyPDFLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfloader/)
- [Langchain RecursiveCharacterTextSplitter](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)


In [59]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
import math

pdf_doc = "./AI_Book.pdf"

# Create pdf data loader
# ADD HERE YOUR CODE
loader = PyPDFLoader(pdf_doc)

# Load and split documents in chunks
# ADD HERE YOUR CODE
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
pages_chunked = loader.load_and_split(text_splitter)

print(pages_chunked[0])

# Function to clean text by removing invalid unicode characters, including surrogate pairs
def clean_text(text):
    # Remove surrogate pairs
    text = re.sub(r'[\ud800-\udfff]', '', text)
    # Keep only alphanumeric characters and spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Remove 'nan' (case insensitive)
    text = re.sub(r'\bnan\b', '', text, flags=re.IGNORECASE)
    return text

pages_chunked_cleaned = []

for doc in pages_chunked:
    # Text bereinigen und Ränder säubern
    cleaned_text = clean_text(doc.page_content).strip()
    
    # NEU: Nur Chunks hinzufügen, die mindestens 10 Zeichen lang sind.
    # Sehr kurze Chunks (z.B. nur "." oder ein einzelnes Wort) verursachen oft NaN-Fehler.
    if len(cleaned_text) > 10 and 'nan' not in cleaned_text.lower(): 
        cleaned_metadata = {}
        for k, v in doc.metadata.items():
            if isinstance(v, float) and math.isnan(v):
                cleaned_metadata[k] = None
            else:
                cleaned_metadata[k] = v
        pages_chunked_cleaned.append(
            Document(page_content=cleaned_text, metadata=cleaned_metadata)
        )

print(f"Number of text chunks after filtering: {len(pages_chunked_cleaned)}")

print(pages_chunked_cleaned[0])

page_content='Aurélien Géron
Hands-on Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2019-05-07T15:51:31+00:00', 'moddate': '2019-05-07T15:51:31+00:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': './AI_Book.pdf', 'total_pages': 510, 'page': 2, 'page_label': '1'}
Number of text chunks after filtering: 1258
page_content='Aurlien Gron
Handson Machine Learning with
ScikitLearn Keras and
TensorFlow
Concepts Tools and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Out

In [49]:
print(pages_chunked_cleaned[1])

page_content='978-1-492-03264-9
[LSI]
Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow
by Aurlien Gron
Copyright  2019 Aurlien Gron. All rights reserved.
Printed in the United States of America.
Published by OReilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.
OReilly books may be purchased for educational, business, or sales promotional use. Online editions are
also available for most titles (http://oreilly.com). For more information, contact our corporate/institutional
sales department: 800-998-9938 or corporate@oreilly.com.
Editor: Nicole Tache
Interior Designer: David Futato
Cover Designer: Karen Montgomery
Illustrator: Rebecca Demarest
June 2019:  Second Edition
Revision History for the Early Release
2018-11-05: First Release
2019-01-24: Second Release
2019-03-07: Third Release
2019-03-29: Fourth Release
2019-04-22: Fifth Release
See http://oreilly.com/catalog/errata.csp?isbn=9781492032649 for release details.' metadata={'producer': 'Antenna

In [50]:
print(f"Number of text chunks: {len(pages_chunked_cleaned)}")

Number of text chunks: 1268


#### Task 3: Store vector embeddings from pdf document to ChromaDB vector database.

**Objective:**

Store vector embeddings into ChromaDB to store knowledge.

**Task Description:**

- create chromadb client
- create chromadb collection
- create langchain chroma db client
- store text document chunks and vector embeddings to vector databases

**Useful links:**

- [Langchain How To](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#initialization-from-client)

In [51]:
from langchain_chroma import Chroma
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

collection_name = "ai_model_book"

# ADD HERE YOUR CODE
# Create a collection
collection = client.get_or_create_collection(name=collection_name)

# ADD HERE YOUR CODE
# Create chromadb
vector_db_from_client = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embedding_model
)

In [60]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(pages_chunked_cleaned))]

# ADD HERE YOUR CODE
vector_db_from_client.add_documents(documents=pages_chunked_cleaned, ids=uuids)

['83a851f7-d755-4b3d-b012-7dec63b44bf4',
 '8648ce3e-b908-4099-80e7-d646c33bafbf',
 'b4a98424-7ef8-43b4-8140-bfa22838e17a',
 '1d173486-18b3-47d6-af08-8f037d3dae31',
 '00237294-6e5e-4b87-949c-ea164dacec18',
 'd0fd650d-c472-43d4-8a2d-d07b3b6358f6',
 'c71cac5b-b5b0-4e71-8a44-33d735b43523',
 '795c65ef-d8a0-4bb1-9cfc-b5ced10a2882',
 'da919777-a70f-431d-9bd6-a281ddf4a0da',
 'ed5162b8-e18c-413a-9809-c51a2f08f8f0',
 'c07c3c0f-7195-49fe-b1a2-113e545a7600',
 'd8502523-2a41-42c3-83c5-a7ea7b256989',
 'a737c8d0-d272-47ca-ae1f-d955e98cacd6',
 '32798f19-aec9-4f3d-b22f-b8b8c8126236',
 '83661ea9-f884-4daa-8ce8-dfc314c10867',
 '8e27dbe7-889c-49cf-92e1-af0657e8740d',
 '20aada68-a63c-49fd-b072-1ee796d40f40',
 '0456bf1b-8f8a-4f58-a719-9dfb99b87ddb',
 'fc365b31-5f4d-4793-9dd6-77bfbbdac4b0',
 '3d68552d-c01d-4012-aa65-a141fd486e76',
 '6a835599-e250-4ea2-9f5e-9db75e477fb7',
 'a645a0fa-bbef-4224-9f19-1896928e0429',
 '24b28b27-944d-4958-a902-0f07c453b74e',
 '8d3f6a0a-c74a-4719-a4f6-3b34a9696e6b',
 'ecdceb4d-c5b9-

In [ ]:
client.count_collections()

1

In [ ]:
# client.delete_collection("ai_model_book")

In [ ]:
# Get and check the content of the collection
collection.get(include=["documents", "metadatas", "embeddings"])

#### Task 4: Access ChromaDB and perform vector search

**Objective:**

Use query to perform vector search against ChromaDB vector database

**Task Description:**

- define query
- run vector search
- print k=3 most similar documents


**Useful links:**

- [Langchain Query ChromaDB](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#search-by-vector)

In [ ]:
search_query = "Types of Machine Learning Systems"

results = vector_db_from_client.similarity_search(search_query, k=3)

for res in results:
    print(res.page_content)
    print(res.metadata)
    print("\n")